In [1]:
import sys

sys.path.append("..")
# from src.datasets.mp16 import ImageDataset
from transformers import Mask2FormerForUniversalSegmentation, AutoImageProcessor
import torch
from torch.utils.data import DataLoader, Dataset
import os
import polars as pl
from PIL import Image
from tqdm import tqdm

HF_CHECKPOINT_NAME = "facebook/mask2former-swin-large-ade-semantic"
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 16
base_img_path = "/mnt/yokoyamalab-nas/gldv2-full/index"

In [2]:
processor = AutoImageProcessor.from_pretrained(HF_CHECKPOINT_NAME)
model = Mask2FormerForUniversalSegmentation.from_pretrained(HF_CHECKPOINT_NAME).to(
    device
)
model = model.eval()
model = torch.compile(model)

The image processor of type `Mask2FormerImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/782 [00:00<?, ?it/s]

In [3]:
df = pl.read_csv("../../datasets/google-landmark/index_set_fixed.csv")
print(len(df))

519263


In [4]:
ids = df["id"].to_numpy().tolist()
paths = [os.path.join(base_img_path, f"{p}.jpg") for p in ids]

In [5]:
class ImageDataset(Dataset):
    def __init__(
        self,
        df: pl.DataFrame,
        img_col: str = "IMG_ID",
        img_base_path: str = "",
    ):
        self.df = df
        self.img_col = img_col
        self.img_base_path = img_base_path

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        df_batch = self.df[index]
        img_id = df_batch[self.img_col].item()
        path = os.path.join(self.img_base_path, f"{img_id}.jpg")

        img = Image.open(path).convert("RGB")
        # target_sizes expects (height, width)
        size = img.size[::-1]

        return {"image": img, "size": size, "img_id": img_id}


def collate_fn(batch):
    images = [b["image"] for b in batch]
    sizes = [b["size"] for b in batch]
    img_id = [b["img_id"] for b in batch]
    inputs = processor(images=images, return_tensors="pt")
    return inputs, sizes, images, img_id

In [6]:
datasets = ImageDataset(df, "id", base_img_path)
loader = DataLoader(datasets, batch_size=8, collate_fn=collate_fn)

In [7]:
# ADE20K class IDs (0-indexed, as returned by HuggingFace)
LANDMARK_SEGMENT_IDS = {
    # ── Core architectural structures ──────────────────────────
    1,  # building, edifice
    25,  # house
    48,  # skyscraper
    84,  # tower
    # ── Bridges & infrastructure ───────────────────────────────
    61,  # bridge, span
    # ── Monuments & sculpture ──────────────────────────────────
    132,  # sculpture
    42,  # column, pillar
    40,  # base, pedestal, stand
    104,  # fountain
    # ── Structural elements ────────────────────────────────────
    53,  # stairs, steps
    59,  # stairway, staircase
    95,  # bannister, balustrade
    38,  # railing, rail
    # ── Contextual / secondary ─────────────────────────────────
    140,  # pier, wharf, dock
    51,  # grandstand
    113,  # waterfall, falls
}

NOISE_DOMINANT_IDS = {
    2,  # sky
    4,  # tree
    9,  # grass
    12,  # person
    20,  # car
    11,  # sidewalk, pavement
    6,  # road, route
    3,  # floor
    26,  # sea
    60,  # river
    46,  # sand
    29,  # field
    16,  # mountain
}


In [8]:
def is_landmark_image(seg_map: torch.Tensor, threshold: float = 0.15) -> bool:
    """
    seg_map: H×W tensor of ADE20K class IDs (0-indexed)
    threshold: minimum fraction of pixels that must be landmark classes
    """
    total_pixels = seg_map.numel()
    landmark_mask = torch.zeros_like(seg_map, dtype=torch.bool)
    for cid in LANDMARK_SEGMENT_IDS:
        landmark_mask |= seg_map == cid

    ratio = landmark_mask.sum().item() / total_pixels
    return {"ratio": ratio, "threshold": threshold, "is_landmark": ratio >= threshold}


# Also save the ratio as a column — lets you retune threshold without re-running

In [9]:
img_landmarks = []

with torch.no_grad(), torch.autocast(device, dtype=torch.bfloat16):
    # batch process
    for img_processed, target_sizes, pil_imgs, ids in tqdm(loader, desc="Extract segments"):
        out = model(**{k: v.to(device) for k, v in img_processed.items()})
        segments = processor.post_process_semantic_segmentation(
            out, target_sizes=target_sizes
        )
        # individual process
        for sgmnt, id in zip(segments, ids):
            res = is_landmark_image(sgmnt)
            res = {"id": id, **res}
            img_landmarks.append(res)

Extract segments: 100%|██████████| 64908/64908 [8:50:17<00:00,  2.04it/s]


In [10]:
img_landmarks_df = pl.DataFrame(img_landmarks)
img_landmarks_df

id,ratio,threshold,is_landmark
str,f64,f64,bool
"""fdf40612109ad174""",0.0,0.15,false
"""5a6cc67c893daea6""",0.0,0.15,false
"""87b88acb68cdc1f1""",0.408953,0.15,true
"""c4ac217ce087b251""",0.404546,0.15,true
"""05f269bf32be9d3e""",0.0,0.15,false
…,…,…,…
"""ace1f52eea7620e9""",0.0,0.15,false
"""f07d97acbc9cac79""",0.247735,0.15,true
"""2a2bdc07e5144f71""",0.138002,0.15,false
